# Lesson 21 Lab — Production Metrics and Alertable Signals

**Puzzle:** Which metric tells you that users are waiting even while GPU utilization looks healthy?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

GPU utilization can remain high while the waiting queue, preemption, cache pressure, or TTFT deteriorates. Operations need request-, scheduler-, cache-, and process-level signals with labels that do not explode cardinality.


## 0. Predict before running

1. Predict which metric families appear after one request.
2. Distinguish counter, gauge, and histogram usage.
3. Write one multi-signal queueing alert.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The lab launches a real localhost vLLM server, generates traffic, scrapes `/metrics`, parses Prometheus samples, and verifies a small required-signal set. It retains names and selected values, not an unbounded scrape.

- A healthy process is not a healthy SLO.
- Metric type determines the correct query.
- Low-cardinality labels are a production requirement.


## 2. Derive the mechanism

Counters accumulate events and should be converted to rates; gauges represent current queue/cache state; histograms support latency distributions over time. Alerts should connect a symptom such as high TTFT to demand, running/waiting requests, cache usage, errors, and saturation. Request IDs and prompts belong in traces/logs under data policy, not metric labels.

### Mechanism at a glance

```mermaid
flowchart LR
  U["client SLO symptoms"] --> H["latency histograms"]
  Q["running + waiting requests"] --> H
  K["KV cache usage"] --> H
  E["errors + preemptions"] --> H
  H --> A["multi-signal alert"]
  A --> R["runbook and rollback"]
```

### Walk it step by step

1. **Start from the SLO.** Choose user-visible TTFT, ITL, completion, and error indicators.
2. **Add causes.** Observe queue, cache, preemption, and process saturation.
3. **Respect metric types.** Rate counters and aggregate histograms over windows.
4. **Test the alert.** Drive known failure states and follow the runbook.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 21
LESSON_TITLE = 'Production Metrics and Alertable Signals'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260833
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | GPU utilization alone |
| Candidate | request, scheduler, cache, latency, and error signals |
| Held constant | same server/model, loopback client, one request, scrape time, and parser |
| Measurements | HTTP status, metric family count, required names, selected values, and unsafe-label scan |
| Evidence | `native-backend` |

**Experiment:** Serve the local model, issue traffic, scrape Prometheus exposition, and validate required metric families.


## 5. Inspect the experiment code

The parser ignores comments and keeps only finite numeric samples. It scans label names for obvious request-content fields and stores a bounded name list for review.

Do not execute until the code matches the frozen table.


In [2]:
payload={"model":str(MODEL),"messages":[{"role":"user","content":"Define one service metric."}],
         "temperature":0.0,"max_tokens":12,"seed":SEED}
probe=run_server_probe(18021,request_payload=payload,scrape_metrics=True); text=probe["metrics_text"]
families=sorted(set(re.findall(r"^# (?:HELP|TYPE) ([^ ]+)",text,flags=re.M)))
names=sorted(set(re.findall(r"^([a-zA-Z_:][a-zA-Z0-9_:]*)",text,flags=re.M)))
groups={"request_success":("vllm:request_success",),"prompt_tokens":("vllm:prompt_tokens",),
        "generation_tokens":("vllm:generation_tokens",),"kv_cache":("vllm:kv_cache_usage_perc",),
        "waiting_requests":("vllm:num_requests_waiting",)}
present={key:any(name in text for name in alternatives) for key,alternatives in groups.items()}
unsafe=re.findall(r'\b(prompt|response|api_key|request_id)="',text,flags=re.I)
metrics={"metrics_status":probe["metrics_status"],"metric_families":len(families),
         "required_present":sum(present.values()),"required_total":len(present),"required_matrix":present,
         "unsafe_label_hits":len(unsafe),"request_succeeded":probe.get("chat_status")==200,
         "sample_names":names[:80],"server_log_tail":SERVER_LOG_TAIL}
analysis=(f"After native traffic, `/metrics` returned HTTP {metrics['metrics_status']} with "
          f"{len(families)} families; {metrics['required_present']}/{metrics['required_total']} required "
          f"groups were found and {len(unsafe)} obvious content/secret labels detected. Thresholds need time series.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Metrics status | 200 |
| Metric families | 86 |
| Required present | 5 |
| Required total | 5 |
| Unsafe labels | 0 |
| Request succeeded | yes |


## 7. Explain the result

After native traffic, `/metrics` returned HTTP 200 with 86 families; 5/5 required groups were found and 0 obvious content/secret labels detected. Thresholds need time series.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`native-backend`**. The named vLLM runtime executed on the recorded GPU/model/workload. The result does not transfer to another version, model, endpoint, or traffic distribution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 21, "title": 'Production Metrics and Alertable Signals', "environment": ENV,
    "evidence_label": 'native-backend', "metrics": metrics,
    "analysis": analysis, "conclusion": 'The native scrape proves observability wiring and available signal names; alert thresholds require time-series workload evidence.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 21,
  "title": "Production Metrics and Alertable Signals",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260833
  },
  "evidence_label": "native-backend",
  "metrics": {
    "metrics_status": 200,
    "metric_families": 86,
    "required_present": 5,
    "required_total": 5,
    "required_matrix": {
      "request_success": true,
      "prompt_tokens": true,
      "generation_tokens": true,
      "kv_cache": true,
      "waiting_requests": true
    },
    "unsafe_label_hits": 0,
    "request_succeeded": true,
    "sample_names": [
      "http_request_duration_highr_seconds_bucket",
      "http_request_duration_highr_seconds_count",
      "http_request_duration_highr_seconds_created",
      "http_request_duration_highr_seconds_sum",
      "http_request_duration_seconds_bu

## 9. Make the bounded decision

> The native scrape proves observability wiring and available signal names; alert thresholds require time-series workload evidence.

**Acceptance/rollback:** Create an alert only when its metric semantics, window, traffic threshold, runbook, and false-positive behavior are tested.

**Failure analysis:** One scrape cannot calculate a rate or quantile and some metrics remain zero without concurrent load. Metric names can change across releases.


## 10. Extend the evidence

Replay sustained and overload traffic, evaluate recording rules, test alerts, and correlate client TTFT with engine histograms and logs.

The full boundary and references are in [`README.md`](README.md).
